In [1]:
!pip install transformers torch

  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ------- -------------------------------- 1.8/10.1 MB 9.1 MB/s eta 0:00:01
   ---------------- ----------------------- 4.2/10.1 MB 11.0 MB/s eta 0:00:01
   -------------------------- ------------- 6.8/10.1 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.1 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 10.9 MB/s  0:00:00
   ---------------------------------------- 0.0/625.2 kB ? eta -:--:--
   ---------------------------------------- 625.2/625.2 kB 4.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ------------------------- -------------- 2.4/3.7 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 11.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ================================
# 1. IMPORT LIBRARIES
# ================================

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


# ================================
# 2. LOAD MODEL
# ================================

print("Loading model... Please wait")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add pad token if missing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.resize_token_embeddings(len(tokenizer))

print("Model loaded successfully ✅\n")


# ================================
# 3. CHATBOT FUNCTION
# ================================

def run_chatbot():

    print("🤖 Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Give short, clear, and accurate answers."
        }
    ]

    while True:
        user_input = input("👤 You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye! Have a great day 😊")
            break

        chat_history.append({"role": "user", "content": user_input})

        # Convert chat to tokens
        inputs = tokenizer.apply_chat_template(
            chat_history,
            return_tensors="pt",
            add_generation_prompt=True
        )

        # Move to device
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # ================================
        # GENERATE RESPONSE (FIXED)
        # ================================
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

        # ================================
        # DECODE RESPONSE
        # ================================
        response = tokenizer.decode(
            outputs[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        )

        # ================================
        # CLEAN RESPONSE
        # ================================
        response = response.strip()
        response = response.split("\n")[0]
        response = response.split("###")[0]

        if "." in response:
            response = response.split(".")[0] + "."

        if len(response) < 5:
            response = "I'm here to help! Could you please ask your question again?"

        print(f"🤖 Chatbot: {response}\n")

        chat_history.append({"role": "assistant", "content": response})


# ================================
# 4. RUN
# ================================

if __name__ == "__main__":
    run_chatbot()

Loading model... Please wait


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully ✅

🤖 Chatbot: Hello! I am your AI assistant. How can I help you today?



👤 You:  Hii


🤖 Chatbot: Hello! How can I assist you today?



...

🤖 Chatbot: The Solar System consists of our own Earth, which orbits the Sun, as well as eight planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.



👤 You:  Where is Pune located?


🤖 Chatbot: Pune is located in the Indian state of Maharashtra.



👤 You:  exit


🤖 Chatbot: Goodbye! Have a great day 😊
